# Survival Modeling

Baseline survival modeling workflow using the merged modeling dataset and readiness artifacts from notebook 10.

This notebook uses Accelerated Failure Time (AFT) models as the primary approach. Because `event_observed` is currently all `1`, interpret this as modeling time to ICU discharge / ICU length of stay rather than a censored survival endpoint.

## Imports and Dependencies

In [7]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parents[0]))

import importlib.util

import numpy as np
import pandas as pd
from lifelines import LogLogisticAFTFitter, LogNormalAFTFitter, WeibullAFTFitter
from lifelines.utils import concordance_index

from config import PROCESSED_DATA_DIR

In [8]:
required_packages = ["lifelines"]
missing_packages = [
    package for package in required_packages
    if importlib.util.find_spec(package) is None
]

if missing_packages:
    message_lines = [
        "Missing modeling packages: " + ", ".join(missing_packages),
        "Install before running this notebook, for example:",
        "    .venv/bin/python -m pip install lifelines",
    ]
    raise ImportError("\n".join(message_lines))

## Load Data

Use the finalized modeling dataset and the model-readiness outputs created by notebook 10.

In [9]:
modeling_df = pd.read_parquet(PROCESSED_DATA_DIR / "modeling_dataset.parquet")
ready_predictor_cols = pd.read_csv(
    PROCESSED_DATA_DIR / "modeling_ready_predictor_columns.csv"
)["predictor_column"].tolist()
ready_numeric_cols = pd.read_csv(
    PROCESSED_DATA_DIR / "modeling_ready_numeric_columns.csv"
)["numeric_predictor_column"].tolist()
ready_categorical_cols = pd.read_csv(
    PROCESSED_DATA_DIR / "modeling_ready_categorical_columns.csv"
)["categorical_predictor_column"].tolist()
split_df = pd.read_csv(PROCESSED_DATA_DIR / "modeling_train_test_split.csv")

print(f"Modeling dataset: {modeling_df.shape}")
print(f"Ready predictors: {len(ready_predictor_cols)}")
print(f"Ready numeric predictors: {len(ready_numeric_cols)}")
print(f"Ready categorical predictors: {len(ready_categorical_cols)}")
print(split_df["split"].value_counts())

Modeling dataset: (94444, 457)
Ready predictors: 443
Ready numeric predictors: 431
Ready categorical predictors: 12
split
train    75380
test     19064
Name: count, dtype: int64


## Build Train/Test Tables

In [10]:
model_df = modeling_df[
    ["subject_id", "stay_id", "duration", "event_observed"] + ready_predictor_cols
].merge(
    split_df[["stay_id", "split"]],
    on="stay_id",
    how="inner",
)

if len(model_df) != len(modeling_df):
    raise ValueError("Split merge changed row count")

if model_df["stay_id"].duplicated().any():
    raise ValueError("Duplicate stay_id rows after split merge")

train_df = model_df[model_df["split"].eq("train")].copy()
test_df = model_df[model_df["split"].eq("test")].copy()

print(f"Model table: {model_df.shape}")
print(model_df["split"].value_counts())

Model table: (94444, 448)
split
train    75380
test     19064
Name: count, dtype: int64


In [11]:
print("Train target summary")
print(train_df[["duration", "event_observed"]].describe())

print()
print("Test target summary")
print(test_df[["duration", "event_observed"]].describe())

if modeling_df["event_observed"].nunique() == 1:
    print()
    print("Note: event_observed is all 1, so AFT predictions can be interpreted as ICU LOS predictions.")

Train target summary
           duration  event_observed
count  75380.000000         75380.0
mean       3.617537             1.0
std        5.373389             0.0
min        0.001250             1.0
25%        1.095943             1.0
50%        1.962668             1.0
75%        3.843134             1.0
max      226.403079             1.0

Test target summary
           duration  event_observed
count  19064.000000         19064.0
mean       3.679404             1.0
std        5.515839             0.0
min        0.003345             1.0
25%        1.097506             1.0
50%        1.975087             1.0
75%        3.941976             1.0
max      121.303368             1.0

Note: event_observed is all 1, so AFT predictions can be interpreted as ICU LOS predictions.


## Preprocess Predictors

Numeric predictors are median-imputed and standardized using train-set statistics. Categorical predictors are filled with `Missing` and one-hot encoded using train-set categories.

In [12]:
numeric_cols = [
    col for col in ready_numeric_cols
    if col in ready_predictor_cols
]
categorical_cols = [
    col for col in ready_categorical_cols
    if col in ready_predictor_cols
]

numeric_medians = train_df[numeric_cols].median()
train_numeric_df = train_df[numeric_cols].fillna(numeric_medians)
test_numeric_df = test_df[numeric_cols].fillna(numeric_medians)

numeric_means = train_numeric_df.mean()
numeric_stds = train_numeric_df.std().replace(0, 1)
train_numeric_df = (train_numeric_df - numeric_means) / numeric_stds
test_numeric_df = (test_numeric_df - numeric_means) / numeric_stds

train_categorical_df = train_df[categorical_cols].astype("object").fillna("Missing").astype(str)
test_categorical_df = test_df[categorical_cols].astype("object").fillna("Missing").astype(str)

train_categorical_dummies_df = pd.get_dummies(
    train_categorical_df,
    columns=categorical_cols,
    drop_first=True,
    dtype=int,
)
test_categorical_dummies_df = pd.get_dummies(
    test_categorical_df,
    columns=categorical_cols,
    drop_first=True,
    dtype=int,
)
test_categorical_dummies_df = test_categorical_dummies_df.reindex(
    columns=train_categorical_dummies_df.columns,
    fill_value=0,
)

print(f"Train numeric matrix: {train_numeric_df.shape}")
print(f"Train categorical matrix: {train_categorical_dummies_df.shape}")
print(f"Test numeric matrix: {test_numeric_df.shape}")
print(f"Test categorical matrix: {test_categorical_dummies_df.shape}")

Train numeric matrix: (75380, 431)
Train categorical matrix: (75380, 69)
Test numeric matrix: (19064, 431)
Test categorical matrix: (19064, 69)


In [13]:
train_model_df = pd.concat(
    [
        train_df[["duration", "event_observed"]].reset_index(drop=True),
        train_numeric_df.reset_index(drop=True),
        train_categorical_dummies_df.reset_index(drop=True),
    ],
    axis=1,
)

test_model_df = pd.concat(
    [
        test_df[["stay_id", "duration", "event_observed"]].reset_index(drop=True),
        test_numeric_df.reset_index(drop=True),
        test_categorical_dummies_df.reset_index(drop=True),
    ],
    axis=1,
)

constant_cols = [
    col for col in train_model_df.columns
    if col not in ["duration", "event_observed"]
    and train_model_df[col].nunique(dropna=False) <= 1
]

if constant_cols:
    train_model_df = train_model_df.drop(columns=constant_cols)
    test_model_df = test_model_df.drop(columns=constant_cols)

print(f"Training matrix: {train_model_df.shape}")
print(f"Test matrix: {test_model_df.shape}")
print(f"Dropped constant encoded columns: {len(constant_cols)}")

Training matrix: (75380, 502)
Test matrix: (19064, 503)
Dropped constant encoded columns: 0


## Fit AFT Models

Fit several penalized AFT distributions and compare them. AFT coefficients are time-ratio effects: `exp(coef) > 1` means longer expected ICU LOS, while `exp(coef) < 1` means shorter expected ICU LOS.

In [14]:
aft_penalizer = 0.1

aft_model_specs = {
    "weibull_aft": WeibullAFTFitter(penalizer=aft_penalizer),
    "lognormal_aft": LogNormalAFTFitter(penalizer=aft_penalizer),
    "loglogistic_aft": LogLogisticAFTFitter(penalizer=aft_penalizer),
}

aft_models = {}
fit_summaries = []

for model_name, model in aft_model_specs.items():
    print(f"Fitting {model_name}...")
    model.fit(
        train_model_df,
        duration_col="duration",
        event_col="event_observed",
        show_progress=False,
    )
    aft_models[model_name] = model
    fit_summaries.append(
        {
            "model": model_name,
            "aic": model.AIC_,
            "log_likelihood": model.log_likelihood_,
        }
    )

fit_summary_df = pd.DataFrame(fit_summaries).sort_values("aic")
fit_summary_df

Fitting weibull_aft...
Fitting lognormal_aft...
Fitting loglogistic_aft...


,model,aic,log_likelihood
1,lognormal_aft,281153.900610,-140074.950305
2,loglogistic_aft,282733.581294,-140864.790647
0,weibull_aft,306950.815988,-152973.407994


## Evaluate AFT Models

In [15]:
def evaluate_aft_model(model_name, model, train_model_df, test_model_df):
    train_features_df = train_model_df.drop(columns=["duration", "event_observed"])
    test_features_df = test_model_df.drop(columns=["stay_id", "duration", "event_observed"])

    train_pred_median = model.predict_median(train_features_df).astype(float)
    test_pred_median = model.predict_median(test_features_df).astype(float)

    train_pred_median = train_pred_median.clip(lower=0)
    test_pred_median = test_pred_median.clip(lower=0)

    train_error = train_pred_median - train_model_df["duration"]
    test_error = test_pred_median - test_model_df["duration"]

    return {
        "model": model_name,
        "train_c_index": concordance_index(
            train_model_df["duration"],
            train_pred_median,
            train_model_df["event_observed"],
        ),
        "test_c_index": concordance_index(
            test_model_df["duration"],
            test_pred_median,
            test_model_df["event_observed"],
        ),
        "train_mae_days": train_error.abs().mean(),
        "test_mae_days": test_error.abs().mean(),
        "train_rmse_days": np.sqrt(np.mean(np.square(train_error))),
        "test_rmse_days": np.sqrt(np.mean(np.square(test_error))),
        "train_median_pred_days": train_pred_median.median(),
        "test_median_pred_days": test_pred_median.median(),
    }


evaluation_df = pd.DataFrame(
    [
        evaluate_aft_model(model_name, model, train_model_df, test_model_df)
        for model_name, model in aft_models.items()
    ]
).sort_values(["test_c_index", "test_mae_days"], ascending=[False, True])

evaluation_df

,model,train_c_index,test_c_index,train_mae_days,test_mae_days,train_rmse_days,test_rmse_days,train_median_pred_days,test_median_pred_days
2,loglogistic_aft,0.750427,0.750923,8.407314e+05,2.325072,2.308257e+08,22.473248,1.949815,1.964937
1,lognormal_aft,0.746786,0.747576,1.014615e+02,2.169801,2.727926e+04,5.992100,2.046912,2.055118
0,weibull_aft,0.743663,0.744502,3.842565e+06,2.513187,1.054992e+09,35.290877,2.137321,2.153052


In [16]:
best_model_name = evaluation_df.iloc[0]["model"]
best_aft_model = aft_models[best_model_name]

print(f"Best AFT model by test C-index: {best_model_name}")
evaluation_df[evaluation_df["model"].eq(best_model_name)]

Best AFT model by test C-index: loglogistic_aft


,model,train_c_index,test_c_index,train_mae_days,test_mae_days,train_rmse_days,test_rmse_days,train_median_pred_days,test_median_pred_days
2,loglogistic_aft,0.750427,0.750923,840731.414859,2.325072,2.308257e+08,22.473248,1.949815,1.964937


## Interpret Coefficients

For AFT models, `exp(coef)` is a multiplicative effect on time. For example, `exp(coef)=1.20` means roughly 20% longer predicted ICU LOS for a one-unit increase in the encoded predictor.

In [17]:
coef_summary_df = best_aft_model.summary.reset_index().rename(
    columns={"covariate": "feature"}
)
coef_summary_df["abs_coef"] = coef_summary_df["coef"].abs()
coef_summary_df = coef_summary_df.sort_values("abs_coef", ascending=False)

coef_summary_df[
    ["param", "feature", "coef", "exp(coef)", "p", "abs_coef"]
].head(40)

,param,feature,coef,exp(coef),p,abs_coef
207,alpha_,first_careunit_Neurology,1.749142,5.749669,2.595564e-03,1.749142
203,alpha_,first_careunit_Medicine/Cardiology Intermediate,-1.236703,0.290340,4.239410e-02,1.236703
210,alpha_,first_careunit_Surgery/Vascular/Intermediate,0.931113,2.537331,2.272435e-36,0.931113
501,beta_,Intercept,0.880120,2.411188,0.000000e+00,0.880120
199,alpha_,first_careunit_Med/Surg,-0.825883,0.437848,1.534013e-01,0.825883
209,alpha_,first_careunit_Surgery/Trauma,0.822366,2.275877,2.052808e-03,0.822366
500,alpha_,Intercept,0.586147,1.797052,1.685278e-46,0.586147
202,alpha_,first_careunit_Medicine,0.582366,1.790270,3.328440e-02,0.582366
204,alpha_,first_careunit_Neuro Intermediate,0.470928,1.601479,4.213081e-50,0.470928
142,alpha_,careunit_group_NEURO,0.442801,1.557062,1.589307e-45,0.442801


## Predictions and Risk Groups

In [18]:
test_features_df = test_model_df.drop(columns=["stay_id", "duration", "event_observed"])
test_pred_median = best_aft_model.predict_median(test_features_df).astype(float).clip(lower=0)

test_predictions_df = test_df[
    ["subject_id", "stay_id", "duration", "event_observed"]
].copy()
test_predictions_df["predicted_median_los_days"] = test_pred_median.to_numpy()
test_predictions_df["predicted_los_percentile"] = test_predictions_df[
    "predicted_median_los_days"
].rank(pct=True)

test_predictions_df.head()

,subject_id,stay_id,duration,event_observed,predicted_median_los_days,predicted_los_percentile
2,10000980,39765666,0.497535,1,0.506575,0.026437
3,10001217,37067082,1.118032,1,1.658852,0.369859
4,10001217,34592300,0.948113,1,1.155912,0.142730
6,10001843,39698942,0.825266,1,0.916879,0.078787
8,10002013,39060235,1.314352,1,1.399634,0.244755


In [19]:
risk_group_df = test_predictions_df.copy()
risk_group_df["predicted_los_group"] = pd.qcut(
    risk_group_df["predicted_median_los_days"],
    q=5,
    labels=["shortest", "short", "middle", "long", "longest"],
    duplicates="drop",
)

risk_group_summary_df = (
    risk_group_df
    .groupby("predicted_los_group", observed=True)
    .agg(
        rows=("stay_id", "size"),
        median_observed_los=("duration", "median"),
        mean_observed_los=("duration", "mean"),
        median_predicted_los=("predicted_median_los_days", "median"),
        mean_predicted_los=("predicted_median_los_days", "mean"),
    )
    .reset_index()
)

risk_group_summary_df

,predicted_los_group,rows,median_observed_los,mean_observed_los,median_predicted_los,mean_predicted_los
0,shortest,3813,0.778507,1.098011,1.005890,0.928832
1,short,3813,1.438519,2.207045,1.515305,1.514449
2,middle,3812,1.934589,2.922622,1.964937,1.970914
3,long,3813,2.842731,4.452316,2.688174,2.718365
4,longest,3813,4.950521,7.716825,4.644595,6.675209


## Save Outputs

In [20]:
model_output_dir = PROCESSED_DATA_DIR / "model_outputs"
model_output_dir.mkdir(exist_ok=True)

fit_summary_df.to_csv(model_output_dir / "aft_fit_summary.csv", index=False)
evaluation_df.to_csv(model_output_dir / "aft_metrics.csv", index=False)
coef_summary_df.to_csv(model_output_dir / "aft_best_coefficients.csv", index=False)
test_predictions_df.to_csv(model_output_dir / "aft_test_predictions.csv", index=False)
risk_group_summary_df.to_csv(
    model_output_dir / "aft_predicted_los_group_summary.csv",
    index=False,
)

pd.DataFrame(
    [
        {
            "selected_model": best_model_name,
            "penalizer": aft_penalizer,
            "train_rows": len(train_model_df),
            "test_rows": len(test_model_df),
            "encoded_predictors": train_model_df.shape[1] - 2,
        }
    ]
).to_csv(model_output_dir / "aft_selected_model.csv", index=False)

print(f"Saved AFT model outputs to {model_output_dir}")

Saved AFT model outputs to /Users/brandonwu/Documents/ICU_LOS_Survival_Analysis/data/processed/model_outputs


## Modeling Notes

In [21]:
print("Modeling notes:")
print("- AFT is the primary model here because the target is ICU LOS and event_observed is all 1.")
print("- Interpret exp(coef) as a multiplicative effect on predicted ICU LOS.")
print("- Compare AFT distributions using C-index, MAE/RMSE, and clinical plausibility.")
print("- If you later add censoring, AFT remains valid, but interpretation should be updated accordingly.")

Modeling notes:
- AFT is the primary model here because the target is ICU LOS and event_observed is all 1.
- Interpret exp(coef) as a multiplicative effect on predicted ICU LOS.
- Compare AFT distributions using C-index, MAE/RMSE, and clinical plausibility.
- If you later add censoring, AFT remains valid, but interpretation should be updated accordingly.
